## Run SUMMA model ##

Run the SUMMA base model, with the model runs managed by Snakemake

NOTE that the file manager will need to be updated for this workflow to run!!!
Further testing is needed.


### Write snakemake file ###

This block of code writes the snakemake file

In [1]:

from pathlib import Path
import sys
import pysumma as ps
# Import local packages
sys.path.append(str(Path('../').resolve()))


from scripts import gpep_to_summa_utils as gts_utils
config = {}
config['summa_forcing_dir'] = Path('/Users/dcasson/Data/gpep/chena/forcing/summa/')

ens, _ = gts_utils.build_ensemble_list(config['summa_forcing_dir'])
file_paths = gts_utils.list_files_in_subdirectory(config['summa_forcing_dir'])

#Filter files remove those that end in .txt
file_paths = [file for file in file_paths if not file.endswith('.txt')]

forcing_file_list = []
for ens_member in ens:
    # write a new file containing the forcing files which correspond to the ensemble member
    forcing_list_file = f'{config["summa_forcing_dir"]}/forcing_files_{ens_member}.txt'
    with open(forcing_list_file, 'w') as f:
        for file in file_paths:
            #if ensemble member in first three characters of file name
            file_path = Path(file)
            if str(file_path.parent) == ens_member:
                f.write(f'{file}.nc\n')
    forcing_file_list.append(forcing_list_file)


print(ens)
print(file_paths)


{'004', '006', '008', '009', '001', '002', '005', '010', '003', '007'}
['007/rf_static_boxcox_low_predictors_ensMember_007', '009/rf_static_boxcox_low_predictors_ensMember_009', '008/rf_static_boxcox_low_predictors_ensMember_008', '001/rf_static_boxcox_low_predictors_ensMember_001', '006/rf_static_boxcox_low_predictors_ensMember_006', '010/rf_static_boxcox_low_predictors_ensMember_010', '003/rf_static_boxcox_low_predictors_ensMember_003', '004/rf_static_boxcox_low_predictors_ensMember_004', '005/rf_static_boxcox_low_predictors_ensMember_005', '002/rf_static_boxcox_low_predictors_ensMember_002']


In [2]:
%%writefile ../rules/run_pysumma_snakemake_test.smk
"""

Snakemake file to run the base SUMMA simulations.

The model simulation is chunked by GRU to allow for parallelization on a cluster.

The chunks of GRUs are defined by the user

"""

from pathlib import Path
import sys
import pysumma as ps
# Import local packages
sys.path.append(str(Path('../').resolve()))

sys.path.append('../')
import gpep_to_summa_utils as gts_utils

# UPDATE LOCAL SUMMA PATH
config['summa_exe'] = '/home/x-dcasson/GitRepos/summa/bin/summa.exe'
config['summa_forcing_dir'] = Path('/anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa_ens/')

# Resolve all file paths and directories in the config file
config['file_manager'] = '/anvil/projects/x-ees240082/dcasson/gpep/bow/summa/settings/fileManager.txt'
config['summa_output_dir'] = '/anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa_output/'
config['case_name'] = 'bow'
config['run_suffix'] = 'base'


ens_set, _ = gts_utils.build_ensemble_list(config['summa_forcing_dir'])
file_paths = gts_utils.list_files_in_subdirectory(config['summa_forcing_dir'])
ens = list(ens_set)

#Filter files remove those that end in .txt
file_paths = [file for file in file_paths if not file.endswith('.txt')]

forcing_file_list = []
for ens_member in ens:
    # write a new file containing the forcing files which correspond to the ensemble member
    forcing_list_file = f'{config["summa_forcing_dir"]}/forcing_files_{ens_member}.txt'
    with open(forcing_list_file, 'w') as f:
        for file in file_paths:
            #if ensemble member in first three characters of file name
            file_path = Path(file)
            if str(file_path.parent) == ens_member:
                f.write(f'{file}.nc\n')
    forcing_file_list.append(forcing_list_file)

rule run_summa_base_simulations:
    input:
        expand(Path(config['summa_output_dir'],f"{config['case_name']}_{{ens_member}}_timestep.nc"),ens_member=ens)

rule run_summa_ensemble_simulations:
    input:
        file_manager = Path(config['file_manager']),
        forcing_file = lambda wildcards: forcing_file_list[ens.index(wildcards.ens_member)]
    output:
        summa_chunked_output = Path(config['summa_output_dir'],f"{config['case_name']}_{{ens_member}}_timestep.nc")
    params:
        summa_exe = config['summa_exe'],
        run_suffix = lambda wildcards: wildcards.ens_member,
    run:
        print(input.forcing_file)
        sim = ps.Simulation(params.summa_exe, input.file_manager)
        forcing_ds = xr.open_dataset(input.forcing_file)
        sim.assign_forcing_file(params.run_suffix, forcing_ds)
        sim.run(run_suffix=str(params.run_suffix))


        

Overwriting ../rules/run_pysumma_snakemake_test.smk


In [ ]:
import pysumma as ps
summa_exe = '/Users/dcasson/GitHub/summa/bin/summa.exe'
file_manager = '/Users/dcasson/Data/yukon_esp/summa/settings/fileManagerSWE.txt'
forcing_file = '/Users/dcasson/Data/yukon_esp/summa/forcing/summa/forcing_files_001.txt'

sim = ps.Simulation(summa_exe, file_manager)


AttributeError: 'Simulation' object has no attribute 'assign'

In [1]:
! snakemake --unlock -s ../rules/run_pysumma_snakemake_test.smk #--configfile ../config/gpep_to_summa_tuolumne_config.yaml
! snakemake -s ../rules/run_pysumma_snakemake_test.smk --cores 8 #--configfile ../config/gpep_to_summa_tuolumne_config.yaml


Unlocking working directory.
Building DAG of jobs...
Using shell: /usr/local/bin/bash
Provided cores: 8
Rules claiming more threads will be scaled down.
Job stats:
job                               count    min threads    max threads
------------------------------  -------  -------------  -------------
run_summa_base_simulations            1              1              1
run_summa_ensemble_simulations        2              1              1
total                                 3              1              1

Select jobs to execute...

[Fri May  9 17:43:15 2025]
rule run_summa_ensemble_simulations:
    input: /anvil/projects/x-ees240082/dcasson/gpep/bow/summa/settings/fileManager.txt, /anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa_ens/forcing_files_001.txt
    output: /anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa_sim/bow_001_timestep.nc
    jobid: 1
    reason: Missing o

In [15]:
%%writefile ../rules/run_fsca_simulations.smk
"""

Snakemake file to calculate fSCA based on SUMMA model simulations.

"""

from pathlib import Path
import sys
# Import local packages
sys.path.append(str(Path('../').resolve()))
sys.path.append(str(Path('../../').resolve()))
sys.path.append(str(Path('../../snow_dist').resolve()))
from scripts import ss_utils
from snow_dist import summa_simulation

# Resolve all file paths and directories in the config file
config['summa_output_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/output'
config['working_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/output'
config['summa_interim_dir'] = '/Users/drc858/Data/gpep/RF_ens/summa/interim'
config['fsca_output_dir'] =  '/Users/drc858/Data/gpep/RF_ens/summa/output/fsca'
config_file_path = '/Users/drc858/GitHub/snow_dist/settings/config_summa_model_tuolumne.yaml'

#Read first raw forcing file for easymore remapping
summa_result_filepaths = list(Path(config['summa_output_dir']).glob('*.nc'))
summa_filenames = [Path(filepath).name for filepath in summa_result_filepaths]
print(summa_filenames)

fsca_simulation = summa_simulation.SummaSimulation(config_file_path)

# Prepare summa model results for fsca
# This includes summing the input and output snowpack variables
rule run_fsca_model:
    input:
        expand(Path(config['summa_interim_dir'], "{filenames}"))

rule prepare_fsca_model_input:
    input:
        summa_result = Path(config['summa_output_dir'],"{filenames}")
    output:
        fsca_input = Path(config['summa_interim_dir'],"{filenames}")
    run:
        summa_ds = xr.open_dataset(input.input_summa_result)
        fsca_ds = summa_fsca_model.prepare_summa_fsca_input(summa_ds)
        fsca_ds.to_netcdf(output.fsca_input)

# Run fsca settings
rule run_fsca_simulations:
    input:
        prepped_files = expand(Path(config['summa_interim_dir'], "{filenames}"), filenames=summa_filenames)
    output:
        fsca_result = Path(config['fsca_output_dir'], "{filenames}")
    run:
        sim = summa_simulation.SummaSimulation(config)
        summa_fsca_model.run_summa_fsca_model(input.prepped_files,output.fsca_result)



Overwriting ../rules/run_fsca_simulations.smk


In [16]:
#-s ../rules/run_pysumma_snakemake.smk --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml 
! snakemake -s ../rules/run_fsca_simulations.smk --cores 8 --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml

ImportError in file /Users/drc858/GitHub/gpep_to_summa_snakemake/workflow/rules/run_fsca_simulations.smk, line 13:
cannot import name 'ss_utils' from 'scripts' (unknown location)
  File "/Users/drc858/GitHub/gpep_to_summa_snakemake/workflow/rules/run_fsca_simulations.smk", line 13, in <module>


In [2]:
%%writefile ../rules/run_pysumma_snakemake_test.smk
from pathlib import Path
import sys
import pysumma as ps
import xarray as xr
import shutil

# Add scripts to path
workflow_dir = Path(workflow.basedir).parent
scripts_path = workflow_dir / "scripts"
sys.path.append(str(scripts_path))
import gpep_to_summa_utils as gts_utils

# ---------------------------------------
# Helper to sanitize paths
# ---------------------------------------
def clean_path(p):
    return Path(str(p).replace(" ", "").replace("\n", "").replace("\t", ""))

# ---------------------------------------
# Load configuration and resolve paths
# ---------------------------------------
config = gts_utils.resolve_paths(config)
CASE_NAME         = config['case_name']
GPEP_FORCING_DIR  = clean_path(config['gpep_forcing_dir'])
SUMMA_OUTPUT_DIR  = clean_path(config['summa_output_dir'])
FILE_MANAGER      = clean_path(config['file_manager'])
SUMMA_EXE         = clean_path(config['summa_exe'])
SUMMA_FORCING_DIR = clean_path(config['summa_forcing_dir'])

print(f"GPEP_FORCING_DIR: {GPEP_FORCING_DIR}")
print(f"SUMMA_OUTPUT_DIR: {SUMMA_OUTPUT_DIR}")
print(f"CASE_NAME: {CASE_NAME}")
print(f"FILE_MANAGER: {FILE_MANAGER}")
print(f"SUMMA_EXE: {SUMMA_EXE}")
print(f"SUMMA_FORCING_DIR: {SUMMA_FORCING_DIR}")

# ---------------------------------------
# Ensemble members (subdirectories)
# ---------------------------------------
ens, _ = gts_utils.build_ensemble_list(SUMMA_FORCING_DIR)
ens = [member.replace(" ", "").replace("\n", "").replace("\t", "") for member in ens if member]

print(f"Ensemble members: {ens}")

# ---------------------------------------
# List forcing files (cleaned, Path objects)
# ---------------------------------------
file_paths = gts_utils.list_files_in_subdirectory(SUMMA_FORCING_DIR, filenames_only=True)
file_paths = [clean_path(fp) for fp in file_paths]
print(f"File paths: {[str(fp) for fp in file_paths]}")

# ---------------------------------------
# Create forcing_files_<member>.txt for SUMMA
# ---------------------------------------
forcing_file_dict = {}
for member_id in ens:
    member_id_clean = member_id.replace(" ", "").replace("\n", "").replace("\t", "")
    forcing_list_path = clean_path(SUMMA_FORCING_DIR / f"forcing_files_{member_id_clean}.txt")
    print(f"Forcing list path: {forcing_list_path}")
    forcing_list_path.parent.mkdir(parents=True, exist_ok=True)

    with forcing_list_path.open('w') as fh:
        for fp in file_paths:
            if fp.parts[0].strip() == member_id_clean:
                full_path = Path(*[part.strip() for part in fp.parts]).with_suffix(".nc")
                fh.write(full_path.as_posix().strip() + "\n")

    forcing_file_dict[member_id_clean] = forcing_list_path

# ---------------------------------------
# Snakemake Rules
# ---------------------------------------
rule all:
    input:
        expand(str(SUMMA_OUTPUT_DIR / (CASE_NAME + "_{ens_member}_timestep.nc")), ens_member=ens)

rule run_summa_ensemble_simulations:
    input:
        file_manager = FILE_MANAGER,
        forcing_file = lambda wildcards: str(forcing_file_dict[wildcards.ens_member])
    output:
        summa_chunked_output = str(SUMMA_OUTPUT_DIR / (CASE_NAME + "_{ens_member}_timestep.nc"))
    params:
        summa_exe = SUMMA_EXE,
        forcing_path = SUMMA_FORCING_DIR,
        forcing_list_name = lambda wildcards: f"forcing_files_{wildcards.ens_member}.txt",
        run_suffix = lambda wildcards: wildcards.ens_member
    run:
        print(f"Running member: {wildcards.ens_member}")
        print(f"Forcing file: {input.forcing_file}")

        sim = ps.Simulation(params.summa_exe, input.file_manager)

        sim_config = {
            'manager': {
                'forcingPath': str(params.forcing_path + '/' + params.run_suffix),
                'forcingListFile': str(params.forcing_list_name)
            }
        }

        sim.apply_config(sim_config)
        sim = ps.Simulation(params.summa_exe, input.file_manager)
        forcing_list_input = Path(params.forcing_path,params.forcing_list_name)
        forcing_list_output = Path(Path(input.file_manager).parent,'.pysumma',params.run_suffix,params.forcing_list_name)

        shutil.copy(forcing_list_input,forcing_list_output)
        
        sim.run(run_suffix=str(params.run_suffix))
        print(sim.status)

Overwriting ../rules/run_pysumma_snakemake_test.smk


In [2]:
import pysumma as ps
from pathlib import Path
import xarray as xr
import shutil
summa_exe = '/home/x-dcasson/GitRepos/summa/bin/summa.exe'
file_manager = '/anvil/projects/x-ees240082/dcasson/gpep/bow/summa/settings/fileManager.txt'
forcing_path = '/anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa/forcing/'
forcing_list_name = 'forcing_files_002.txt'
run_suffix = '002'

sim = ps.Simulation(summa_exe, file_manager)
print(sim.manager)
sim_config = {
    'manager': {
        'forcingPath': str(forcing_path + '/' + run_suffix),
        'forcingListFile': str(forcing_list_name)
    }
}
forcing_list_input = Path(forcing_path,forcing_list_name)

forcing_list_output = Path(Path(file_manager).parent,'.pysumma',run_suffix,forcing_list_name)
print(forcing_list_input)
print(forcing_list_output)
shutil.copy(forcing_list_input,forcing_list_output)
sim.apply_config(sim_config)

#forcing_ds = xr.open_dataset(forcing_file)
#sim.assign_forcing_file(run_suffix, forcing_ds)
#sim.config_path = Path(settings_path,'.pysumma', run_suffix)
#sim._write_file_manager()
sim.run(run_suffix=run_suffix)

print(sim.status)




controlVersion                       'SUMMA_FILE_MANAGER_V3.0.0'
simStartTime                         '2008-10-01 00:00'
simEndTime                           '2008-10-05 23:00'
tmZoneInfo                           'utcTime'
outFilePrefix                        'bow'
settingsPath                         '/anvil/projects/x-ees240082/dcasson/gpep/bow/summa/settings/'
forcingPath                          '/anvil/projects/x-ees240082/dcasson/gpep/bow/summa/forcing/'
outputPath                           '/anvil/projects/x-ees240082/dcasson/gpep/bow/summa/output/'
initConditionFile                    'coldState.nc'
attributeFile                        'attributes.nc'
trialParamFile                       'trialParams.nc'
forcingListFile                      'forcingFileList.txt'
decisionsFile                        'modelDecisions.txt'
outputControlFile                    'outputControl.txt'
globalHruParamFile                   'localParamInfo.txt'
globalGruParamFile                   'basinPa

In [10]:
from pathlib import Path
import sys
import pysumma as ps
import xarray as xr
import shutil
import os

def build_ensemble_list(summa_forcing_dir):
    """
    Returns a list of ensemble member IDs based on subdirectories in the SUMMA forcing dir.
    Only returns folder names that are all digits (e.g., '001', '002').
    """
    summa_forcing_dir = Path(summa_forcing_dir)
    members = sorted([
        p.name for p in summa_forcing_dir.iterdir()
        if p.is_dir() and p.name.strip().isdigit()
    ])
    return members, None  # match original return signature

def list_files_in_subdirectory(directory, suffix_to_remove='.nc', filenames_only=False):
    """
    List all files recursively, optionally removing suffix from filenames.
    Returns clean Path objects. Optionally, return only filenames (no subdirectories).
    
    Args:
        directory (str or Path): Root directory to search.
        suffix_to_remove (str): Suffix to strip from filenames (default: '.nc').
        filenames_only (bool): If True, return only filenames (no subdirectories). Default is False.
    
    Returns:
        List[Path] or List[str]: Cleaned Path objects or filenames as strings.
    """
    path = Path(directory)
    file_paths = []

    for file in path.glob('**/*.nc'):
        if file.is_file():
            rel_path = file.relative_to(path)
            # Remove suffix safely only from the filename part
            if rel_path.name.endswith(suffix_to_remove):
                cleaned_name = rel_path.name[:-len(suffix_to_remove)]
            else:
                cleaned_name = rel_path.name
            if filenames_only:
                file_paths.append(cleaned_name)
            else:
                cleaned_path = rel_path.with_name(cleaned_name)
                file_paths.append(cleaned_path)
    return file_paths
# ---------------------------------------
# Helper to sanitize paths
# ---------------------------------------
def clean_path(p):
    return Path(str(p).replace(" ", "").replace("\n", "").replace("\t", ""))

# ---------------------------------------
# Load configuration and resolve paths
# ---------------------------------------
CASE_NAME         = 'bow'
SUMMA_FORCING_DIR = Path('/anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa/forcing/')

# ---------------------------------------
# Ensemble members (subdirectories)
# ---------------------------------------
ens, _ = build_ensemble_list(SUMMA_FORCING_DIR)
ens = [member.replace(" ", "").replace("\n", "").replace("\t", "") for member in ens if member]

print(f"Ensemble members: {ens}")

# ---------------------------------------
# List forcing files (cleaned, Path objects)
# ---------------------------------------
file_paths = list_files_in_subdirectory(SUMMA_FORCING_DIR, filenames_only=True)
file_paths = [clean_path(fp) for fp in file_paths]
print(f"File paths: {[str(fp) for fp in file_paths]}")

# ---------------------------------------
# Create forcing_files_<member>.txt for SUMMA
# ---------------------------------------
forcing_file_dict = {}
for member_id in ens:
    member_id_clean = member_id.replace(" ", "").replace("\n", "").replace("\t", "")
    forcing_list_path = clean_path(SUMMA_FORCING_DIR / f"forcing_files_{member_id_clean}.txt")
    print(f"Forcing list path: {forcing_list_path}")
    forcing_list_path.parent.mkdir(parents=True, exist_ok=True)

    with forcing_list_path.open('w') as fh:
        for fp in file_paths:
            if member_id_clean in fp.parts[0].strip():
                full_path = Path(*[part.strip() for part in fp.parts]).with_suffix(".nc")
                fh.write(full_path.as_posix().strip() + "\n")

    forcing_file_dict[member_id_clean] = forcing_list_path

Ensemble members: ['001', '002']
File paths: ['rf_best_regression_static_dynamic_ensMember_002', 'rf_best_regression_static_dynamic_ensMember_001']
Forcing list path: /anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa/forcing/forcing_files_001.txt
Forcing list path: /anvil/projects/x-ees240082/dcasson/gpep/bow/ensemble_generation/rf_best_regression_static_dynamic/summa/forcing/forcing_files_002.txt
